# Occupancy, registers and latency hiding

Why a GPU needs many warps in flight to hide memory latency, how registers, shared memory and block size limit the number of warps per SM, how to compute occupancy by hand and with the CUDA API, and why the fastest kernels often run at low occupancy and hide latency with more work per thread instead.

This notebook holds the code from the lesson **[Occupancy, registers and latency hiding](https://inferenceacademy.com/learn/kern-cuda/occupancy-registers-and-latency-hiding)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## The problem: memory is far away

## Occupancy

## Checking occupancy

The next cell uses `mm.cu`, which you put together yourself from the lesson's snippets (or an earlier lesson). Create it first in a cell that starts with `%%writefile mm.cu`.

In [ ]:
!nvcc -O3 -arch=sm_75 -Xptxas -v -o mm mm.cu 2>&1 | grep -A1 sgemm
# ptxas info : Used 40 registers, 8192 bytes smem, 380 bytes cmem[0] for 'sgemm_reg...'

Snippet from the lesson (add it to your program; not run on its own):

```cpp
int blocks_per_sm;
cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocks_per_sm, sgemm_reg, 256, 0);
printf("blocks per SM: %d, occupancy: %.0f%%\n", blocks_per_sm, 100.0 * blocks_per_sm * 8 / 32);
```

## Controlling register use

Snippet from the lesson (add it to your program; not run on its own):

```cpp
// Per kernel: blocks of at most 256 threads, at least 2 blocks per SM.
__global__ void __launch_bounds__(256, 2) my_kernel(...) { ... }
```

In [ ]:
# For the whole file (blunter):
!nvcc -maxrregcount=64 ...

## Occupancy is a means, not a goal

Snippet from the lesson (add it to your program; not run on its own):

```cpp
float acc = 0.f;
for (int k = 0; k < K; k++) acc += a[k] * b[k];   // each FMA waits for the previous one
```

Snippet from the lesson (add it to your program; not run on its own):

```cpp
float s0 = 0, s1 = 0, s2 = 0, s3 = 0;
for (int k = 0; k < K; k += 4) {
  s0 += a[k] * b[k]; s1 += a[k + 1] * b[k + 1]; s2 += a[k + 2] * b[k + 2]; s3 += a[k + 3] * b[k + 3];
}
float acc = s0 + s1 + s2 + s3;   // (assumes K is a multiple of 4)
```

## Where this shows up in inference

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-cuda/occupancy-registers-and-latency-hiding